# Lesson 3 — Chatbot with Memory using Ollama

## What you will learn
- `MessagesState` pattern — the standard way to build chat graphs
- **Reducers** — how `add_messages` appends instead of replacing
- Connecting a **real LLM** (Ollama, local & free) as a node
- Multi-turn conversations with full history

## Prerequisites
1. Ollama installed → https://ollama.com
2. Run in terminal: `ollama pull llama3`
3. Ollama server must be running (it starts automatically after install)

## Mental model
```
Turn 1:  [HumanMessage("Hi")] → chatbot → [HumanMessage("Hi"), AIMessage("Hello!")]
Turn 2:  [HumanMessage("Hi"), AIMessage("Hello!"), HumanMessage("How are you?")] → chatbot → [..., AIMessage("I'm good!")]
```
The messages list **grows** with every turn — that's the memory!

## Understanding Reducers

In Lesson 1, when a node returned `{"key": new_value}`, it **replaced** the old value.  
With `add_messages`, the behavior changes to **append**:

```python
# Without reducer — replaces
messages: list   # old=[msg1]  →  node returns [msg2]  →  new=[msg2]

# With add_messages reducer — appends
messages: Annotated[list, add_messages]  # old=[msg1]  →  node returns [msg2]  →  new=[msg1, msg2]
```

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
from config import get_ollama_model

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict

## Step 1 — State with Messages Reducer

In [2]:
class ChatState(TypedDict):
    # Annotated[list, add_messages] means:
    # "this list uses add_messages as its update rule"
    messages: Annotated[list, add_messages]

print("State defined — messages will APPEND, not replace")

State defined — messages will APPEND, not replace


## Step 2 — Connect to Ollama

In [5]:
# Make sure Ollama is running and llama3 is pulled
# To change model: replace "llama3.2" with "mistral", "gemma2", etc.
llm = ChatOllama(model=get_ollama_model(), temperature=0.7)

# Quick test — verify Ollama is working
test_response = llm.invoke([HumanMessage(content="Say 'Ollama is working!' and nothing else.")])
print("Ollama test:", test_response.content)

Ollama test: Ollama is working!


## Step 3 — The Chatbot Node

In [6]:
def chatbot_node(state: ChatState) -> dict:
    # The LLM sees the FULL conversation history
    response = llm.invoke(state["messages"])
    # Return only the NEW message — add_messages will append it
    return {"messages": [response]}

## Step 4 — Build, Compile, Visualize

In [ ]:
graph_builder = StateGraph(ChatState)
graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except:
    print("Graph compiled! (visualization unavailable)")

## Step 5 — Single Turn Conversation

In [12]:
result = graph.invoke({
    "messages": [HumanMessage(content="What is LangGraph in one sentence?")]
})

print("AI:", result["messages"][-1].content)

AI: I couldn't find any information on "LangGraph". It's possible that it's a relatively new or obscure term, or it may be a typo or misspelling. If you could provide more context or clarify what LangGraph refers to, I'd be happy to try and help further.


## Step 6 — Multi-Turn Conversation (Memory in Action)

We manually pass the growing history to simulate multi-turn chat.

In [14]:
conversation_history = []

def chat(user_input: str) -> str:
    global conversation_history
    conversation_history.append(HumanMessage(content=user_input))
    result = graph.invoke({"messages": conversation_history})
    conversation_history = result["messages"]
    return conversation_history[-1].content

# Turn 1
print("You: My name is Ahmed.")
reply1 = chat("My name is Ahmed.")
print(f"AI: {reply1}")

You: My name is Ahmed.
AI: Hello Ahmed! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [15]:
# Turn 2 — does the AI remember the name?
print("You: What is my name?")
reply2 = chat("What is my name?")
print(f"AI: {reply2}")

print(f"\nTotal messages in history: {len(conversation_history)}")

You: What is my name?
AI: I remember, your name is Ahmed.

Total messages in history: 4


In [16]:
# Turn 3
print("You: What programming language are we learning about?")
reply3 = chat("What programming language are we learning about?")
print(f"AI: {reply3}")

You: What programming language are we learning about?
AI: We weren't explicitly stated to be learning a specific programming language. Our conversation just started, and I don't have any prior knowledge or context about what programming language you might be interested in learning. Would you like to discuss a particular language or get recommendations on where to start?


## Step 7 — Using a System Prompt

A `SystemMessage` sets the AI's personality/role — it goes **first** in the messages list.

In [17]:
system_msg = SystemMessage(content="You are a helpful Python tutor. Always give short, practical answers with code examples.")

result = graph.invoke({
    "messages": [
        system_msg,
        HumanMessage(content="How do I reverse a list in Python?")
    ]
})
print(result["messages"][-1].content)

**Reversing a List in Python**

Python provides several ways to reverse a list. Here's the most common one:

```python
# Original list
my_list = [1, 2, 3, 4, 5]

# Reversed list using slicing
reversed_list = my_list[::-1]
print(reversed_list)  # Output: [5, 4, 3, 2, 1]
```

In this example, `my_list[::-1]` tells Python to start from the end of the list and move backwards to the beginning, stepping backwards by 1 element each time.

**Alternative Methods**

* Using the built-in `reverse()` method:
```python
my_list = [1, 2, 3, 4, 5]
my_list.reverse()
print(my_list)  # Output: [5, 4, 3, 2, 1]
```
This method modifies the original list in place.

* Using slicing with a step of -1:
```python
my_list = [1, 2, 3, 4, 5]
reversed_list = my_list[-1::-1]
print(reversed_list)  # Output: [5, 4, 3, 2, 1]
```
This method creates a new reversed list without modifying the original.

Choose the method that best fits your use case!


## Key Takeaways

| Concept | Description |
|---------|-------------|
| `Annotated[list, add_messages]` | Appends new messages to history instead of replacing |
| `HumanMessage` | Wraps user input |
| `AIMessage` | The LLM's response |
| `SystemMessage` | Sets AI persona/instructions (goes first) |
| `ChatOllama(model=...)` | Local LLM via Ollama |
| Memory | Just pass the growing `messages` list each turn |

## 🏋️ Exercise
1. Create a **Pirate Chatbot** by adding a `SystemMessage` that says:
   `"You are a pirate. Always respond in pirate language with 'Arrr!'`"
2. Ask it: `"What is machine learning?"`
3. Follow up: `"Give me an example."`
4. Does it stay in pirate character across turns?

In [ ]:
# Your pirate chatbot here
